# 21.6 Rule-based and expert systems

Expert systems make reasoning inspectable: facts trigger rules, rules add facts, and the fired chain explains the answer.

Rule-based expert systems separate knowledge from the inference engine. Forward chaining repeatedly fires rules whose antecedents are already facts and records an audit trace.

Save a copy to Drive to edit.

In [ ]:

import random

import matplotlib.pyplot as plt

random.seed(2106)


def forward_chain(facts, rules, certainty=None, max_rounds=20):
    known = set(facts)
    certainties = dict(certainty or {})
    trace = []
    fired = set()
    fact_counts = [len(known)]
    for _ in range(max_rounds):
        changed = False
        for index, rule in enumerate(rules):
            key = (index, tuple(sorted(known)))
            antecedents = set(rule["if"])
            if antecedents.issubset(known) and key not in fired:
                fired.add(key)
                base = min([certainties.get(item, 1.0) for item in antecedents] or [1.0])
                confidence = base * rule.get("weight", 1.0)
                old_confidence = certainties.get(rule["then"], -1.0)
                if confidence > old_confidence:
                    certainties[rule["then"]] = confidence
                if rule["then"] not in known:
                    known.add(rule["then"])
                    changed = True
                    trace.append((rule["name"], tuple(rule["if"]), rule["then"], confidence))
                    fact_counts.append(len(known))
        if not changed:
            break
    return {"facts": known, "certainty": certainties, "trace": trace, "fact_counts": fact_counts}


def make_rule_ladder():
    return [
        {"name": "D1 triage", "facts": {"fever", "cough"}, "certainty": {"fever": 0.8, "cough": 0.7}, "rules": [{"name": "flu", "if": ["fever", "cough"], "then": "flu", "weight": 0.9}, {"name": "rest", "if": ["flu"], "then": "rest"}, {"name": "hydrate", "if": ["rest"], "then": "hydrate"}], "target": "hydrate"},
        {"name": "D2 symptoms actions", "facts": {"fever", "cough", "aches"}, "certainty": {}, "rules": [{"name": "flu", "if": ["fever", "cough"], "then": "flu"}, {"name": "clinic", "if": ["flu", "aches"], "then": "call_clinic"}, {"name": "rest", "if": ["flu"], "then": "rest"}], "target": "call_clinic"},
        {"name": "D3 priorities conflict", "facts": {"high_amount", "new_account", "known_customer"}, "certainty": {}, "rules": [{"name": "review", "if": ["high_amount", "new_account"], "then": "manual_review", "weight": 0.8}, {"name": "trusted", "if": ["known_customer"], "then": "trusted", "weight": 0.9}, {"name": "expedite", "if": ["trusted"], "then": "expedite", "weight": 0.8}], "target": "manual_review"},
        {"name": "D4 support tickets", "facts": {"login", "vip", "outage", "mobile", "paid"}, "certainty": {}, "rules": [{"name": "auth", "if": ["login"], "then": "auth_team"}, {"name": "incident", "if": ["outage"], "then": "incident_team"}, {"name": "priority", "if": ["vip", "paid"], "then": "priority_queue"}, {"name": "mobile", "if": ["mobile", "auth_team"], "then": "mobile_auth"}], "target": "mobile_auth"},
        {"name": "D5 policy engine", "facts": {"id_verified", "income_ok", "low_debt", "no_fraud", "long_history"}, "certainty": {"id_verified": 0.95, "income_ok": 0.8, "low_debt": 0.75, "no_fraud": 0.9, "long_history": 0.85}, "rules": [{"name": "basic", "if": ["id_verified", "no_fraud"], "then": "basic_clear", "weight": 0.95}, {"name": "credit", "if": ["income_ok", "low_debt"], "then": "credit_clear", "weight": 0.9}, {"name": "history", "if": ["long_history"], "then": "history_clear", "weight": 0.85}, {"name": "eligible", "if": ["basic_clear", "credit_clear", "history_clear"], "then": "eligible", "weight": 0.9}, {"name": "offer", "if": ["eligible"], "then": "offer", "weight": 0.95}], "target": "offer"},
    ]


## The concept, built once (D1)

Forward chaining adds $c$ when $A\subseteq Facts$. The lesson chain grows $\{fever,cough\}$ to flu, rest, and hydrate, and the certainty factor is $min(0.8,0.7)	imes0.9=0.63$.

In [ ]:

facts = {"fever", "cough"}
certainty = {"fever": 0.8, "cough": 0.7}
rules = [{"name": "flu", "if": ["fever", "cough"], "then": "flu", "weight": 0.9}, {"name": "rest", "if": ["flu"], "then": "rest"}, {"name": "hydrate", "if": ["rest"], "then": "hydrate"}]

d1 = forward_chain(facts, rules, certainty)
print(d1["fact_counts"])
print(d1["certainty"]["flu"])
print(d1["trace"])


The lesson's exact fixed-point count is $2	o3	o4	o5$, and the certainty factor is $0.63$. The trace explains each added fact.

In [ ]:

assert d1["fact_counts"] == [2, 3, 4, 5]
assert abs(d1["certainty"]["flu"] - 0.63) < 1e-12
assert "hydrate" in d1["facts"]


## The dataset ladder

D1-D5 grow from a tiny triage system to a longer policy engine with certainty factors and an explanation trail.

In [ ]:

rule_ladder = make_rule_ladder()

for rung in rule_ladder:
    print(rung["name"], "facts", len(rung["facts"]), "rules", len(rung["rules"]), "target", rung["target"])


## Run the same method across D1-D5

Every rung uses the same forward chaining method. The metric records target correctness, rules fired, and fixed-point fact count.

In [ ]:

rule_results = []
for rung in rule_ladder:
    result = forward_chain(rung["facts"], rung["rules"], rung.get("certainty"))
    correct = rung["target"] in result["facts"]
    rule_results.append({
        "name": rung["name"],
        "rules": len(rung["rules"]),
        "fired": len(result["trace"]),
        "facts": len(result["facts"]),
        "correct": correct,
        "result": result,
    })

for row in rule_results:
    print(row["name"], row["rules"], row["fired"], row["facts"], row["correct"])


## Results visualization

The panels show fired-rule traces. The summary curve compares rulebase size with fired rules and final fact count.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], rule_results):
    text = "\n".join(f"{name}: {ants} -> {then} ({confidence:.2f})" for name, ants, then, confidence in row["result"]["trace"][:5])
    ax.text(0.03, 0.95, text, va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["rules"] for row in rule_results], [row["fired"] for row in rule_results], marker="o", label="fired")
ax.plot([row["rules"] for row in rule_results], [row["facts"] for row in rule_results], marker="x", label="facts")
ax.set_xlabel("rule count")
ax.set_title("fixed point growth")
ax.legend()
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: ignoring explanation paths. On D5, a bare offer fact is less useful than the full antecedent, consequent, and certainty trace.

In [ ]:

hard = rule_ladder[-1]
fixed = forward_chain(hard["facts"], hard["rules"], hard.get("certainty"))
wrong = "offer" in fixed["facts"]

print("wrong bare answer", wrong)
print("trace length", len(fixed["trace"]))
print("offer certainty", fixed["certainty"].get("offer"))
print("trace", fixed["trace"])


## Evaluate it + Practice

- Metric: target correctness, rules fired, and fixed-point fact count, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add one rule after hydrate and predict the new fact-count path.

Practice: Lower cough certainty and recompute the flu certainty formula.

Practice: Remove trace recording and list what cannot be audited.